In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import seaborn as sns
from IPython.display import display

In [2]:
TASK = Path.cwd().parent
ROOT = TASK.parent
INPUT, OUTPUT = TASK/'input', TASK/'output'
CONFIG = json.loads((ROOT/'config.json').read_text())
Y, SEED = CONFIG['last_year'], CONFIG['seed']
EMPIRICAL = 'empirical_complete'
SELECTED = [EMPIRICAL] + CONFIG['models']
LABELS = {'empirical_complete':'DBLP empirical', 'arw4':'ARW4', 'lognormal_grw':'GRW',
          'ar1_s':'AR(1)', 'hurdle_ar1_s_p':'Hurdle AR(1)', 'ar6_6':'AR(6,6)', 'se_hurdle_s':'Self-exciting hurdle'}
palette = sns.color_palette('colorblind', 6)
COLORS = {EMPIRICAL:'#222222', 'lognormal_grw':palette[0], 'ar1_s':palette[1],
          'hurdle_ar1_s_p':palette[4], 'se_hurdle_s':palette[2], 'ar6_6':palette[5], 'arw4':palette[3]}
LINES = dict(zip(SELECTED, ['-', ':', '--', '-.', ':', '--', '-']))
MARKERS = dict(zip(SELECTED, ['o', 'P', 's', '^', 'D', 'X', 'v']))
sns.set_theme(style='ticks', context='paper',
              rc={'font.size':9, 'axes.labelsize':9, 'axes.titlesize':10,
                  'legend.fontsize':8, 'pdf.fonttype':42, 'ps.fonttype':42, 'savefig.dpi':350})

In [3]:
errors = pd.read_csv(OUTPUT/'annual_and_increment_errors.csv')
rank_errors = pd.read_csv(OUTPUT/'rank_errors.csv')
path_cdfs = pd.read_csv(OUTPUT/'path_ecdf_knots.csv')
curve_scores = pd.read_csv(OUTPUT/'curve_agreement.csv')
reference = errors[['domain', 'scale', 'statistic', 'age', 'empirical']].drop_duplicates().rename(columns={'empirical':'value'})
reference['source'] = EMPIRICAL
simulated = errors[['source', 'domain', 'scale', 'statistic', 'age', 'simulated']].rename(columns={'simulated':'value'})
curves = pd.concat([reference, simulated], ignore_index=True)

In [4]:
def add_legend(fig, selected):
    handles = [Line2D([], [], color=COLORS[tag], linestyle=LINES[tag], marker=MARKERS[tag],
                      markersize=3, label=LABELS[tag]) for tag in selected]
    fig.legend(handles=handles, loc='upper center', ncol=4, frameon=False, bbox_to_anchor=(.5, 1.01))
def save_figure(fig, name):
    for extension in ['pdf', 'png']:
        fig.savefig(OUTPUT/f'{name}.{extension}', bbox_inches='tight', facecolor='white')
    plt.close(fig)
def draw_curves(ax, frame, selected, x, y):
    for source in selected:
        data = frame.loc[frame.source.eq(source)].sort_values(x)
        sns.lineplot(data=data, x=x, y=y, ax=ax, color=COLORS[source], linestyle=LINES[source],
                     marker=MARKERS[source], markersize=3, markevery=3,
                     linewidth=1.3, estimator=None, errorbar=None, legend=False)
    sns.despine(ax=ax)

In [5]:
# alternate views preserve GRW tails and make the other models readable
for scale in ['raw', 'offset_log']:
    for view, selected in [('all', SELECTED), ('without_grw', [s for s in SELECTED if s!='lognormal_grw'])]:
        fig, axes = plt.subplots(1, 3, figsize=(9.0, 3.4))
        for ax, statistic in zip(axes, ['mean', 'median', 'variance']):
            frame = curves.loc[curves.domain.eq('annual') & curves.scale.eq(scale) & curves.statistic.eq(statistic)]
            draw_curves(ax, frame, selected, 'age', 'value')
            ax.axvspan(0, 5, color='.9', alpha=.45, zorder=0)
            ax.set(xlabel='Career year', ylabel=statistic.capitalize(), title=statistic.capitalize(), xlim=(0, Y))
            if scale=='raw' and view=='all':
                ax.set_yscale('symlog', linthresh=1)
                ax.set_ylabel(f'{statistic.capitalize()} (symlog)')
        add_legend(fig, selected)
        fig.subplots_adjust(left=.08, right=.98, bottom=.20, top=.74, wspace=.40)
        save_figure(fig, f'annual_curves_{scale}_{view}')

In [6]:
# annual changes are labeled by destination career year
for scale in ['raw', 'offset_log']:
    fig, axes = plt.subplots(1, 3, figsize=(9.0, 3.4))
    for ax, statistic in zip(axes, ['mean', 'median', 'variance']):
        frame = curves.loc[curves.domain.eq('increment') & curves.scale.eq(scale) & curves.statistic.eq(statistic)]
        draw_curves(ax, frame, SELECTED, 'age', 'value')
        ax.axvspan(1, 6, color='.9', alpha=.45, zorder=0)
        ax.set(xlabel='Destination career year', ylabel=statistic.capitalize(), title=statistic.capitalize(), xlim=(1, Y))
        if scale=='raw':
            ax.set_yscale('symlog', linthresh=1)
            ax.set_ylabel(f'{statistic.capitalize()} (symlog)')
    add_legend(fig, SELECTED)
    fig.subplots_adjust(left=.08, right=.98, bottom=.20, top=.74, wspace=.40)
    save_figure(fig, f'increment_curves_{scale}')

In [7]:
# path summaries compare order-dependent structure across generated years 6--20
titles = {'active_fraction':'Active-year fraction', 'longest_observed_zero_run':'Longest observed\ninactive run',
          'max_three_year_share':'Maximum three-year\noutput share', 'normalized_total_variation':'Relative annual variation',
          'peak_age':'Year of maximum output', 'log1p_trend_slope':'Within-path log1p trend'}
fig, axes = plt.subplots(2, 3, figsize=(9.0, 6.0))
for ax, (descriptor, title) in zip(axes.flat, titles.items()):
    frame = path_cdfs.loc[path_cdfs.window.eq('common_generated') & path_cdfs.descriptor.eq(descriptor)]
    for source in SELECTED:
        data = frame.loc[frame.source.eq(source)].sort_values('x')
        sns.lineplot(data=data, x='x', y='cdf', ax=ax, color=COLORS[source], linestyle=LINES[source],
                     linewidth=1.3, estimator=None, errorbar=None, legend=False)
    ax.set(xlabel=title, ylabel='Cumulative probability', ylim=(0, 1.02))
    sns.despine(ax=ax)
add_legend(fig, SELECTED)
fig.subplots_adjust(left=.08, right=.98, bottom=.10, top=.87, hspace=.45, wspace=.35)
save_figure(fig, 'path_features_common_generated')

In [8]:
caption = (
    'Annual population curves compare the complete empirical cohort with model simulations. '
    'Raw and offset-log views use the same underlying values and ddof=0 variances. '
    'Shading marks years affected by the six-year empirical prefix of AR(6,6); other models generate from year 1. '
    'Annual-curve common-window scores use years 6--20; increment scores use destination years 7--20 '
    'so both endpoints are generated for all models. '
    'NRMSE is RMSE divided by the root mean square of the empirical curve over the same years. '
    'Path-feature panels use years 6--20 and connect compact ECDF knots; numerical distribution distances '
    'use all defined observations. Zero-output paths have undefined concentration and peak age and are excluded '
    'from those two descriptors only. Inactive runs are the longest observed within-window runs and may be '
    'truncated at boundaries; first maxima win ties. '
    'Population curves and unpaired distributions of path features describe different aspects of agreement, '
    'not predictions for identified researchers. All comparisons are in-sample and fit cohorts differ across models.'
)
(OUTPUT/'trajectory_figure_notes.txt').write_text(caption+'\n')
display(curve_scores.loc[curve_scores.window.eq('common_generated') & curve_scores.domain.eq('annual')
                         & curve_scores.scale.eq('raw') & curve_scores.statistic.isin(['mean', 'median', 'variance']),
                         ['source', 'statistic', 'rmse', 'nrmse']])

,source,statistic,rmse,nrmse
150,arw4,mean,5.569519e-01,7.512927e-02
152,lognormal_grw,mean,5.759708e+03,7.769480e+02
154,ar1_s,mean,3.907441e-01,5.270890e-02
156,hurdle_ar1_s_p,mean,9.855248e-01,1.329410e-01
158,ar6_6,mean,1.077294e+00,1.453202e-01
160,se_hurdle_s,mean,2.190231e-01,2.954482e-02
162,arw4,median,4.663862e-01,8.386476e-02
164,lognormal_grw,median,1.053371e+00,1.894153e-01
166,ar1_s,median,1.713348e+00,3.080913e-01
168,hurdle_ar1_s_p,median,1.079333e+00,1.940839e-01
